# **Projeto PCD: K-means 1D (naive) com Paralelização Progressiva**

## Profs. Álvaro e Denise (Turma 2024)
**Alunos:**
1. Anna Clara Medina Roissmann
2. Marcos Alves de Aquino
3. Tasso Eliézer Daflon Cicarino Canellas

---
### Objetivo
Este notebook apresenta a implementação do algoritmo K-means unidimensional para agrupar dados de temperatura. A evolução parte de uma versão sequencial simples (naive), passa por OMP e culmina na paralelização distribuída com MPI.

**Dataset:** `city_temperature.csv` (Coluna AvgTemperature)

In [ ]:
import pandas as pd

# Specify dtype for column 2 to handle mixed types
df = pd.read_csv('/content/city_temperature.csv', low_memory=False)

# Extrair apenas a coluna de temperatura e remover -99 (erro)
data_clean = df[df['AvgTemperature'] != -99]['AvgTemperature']

# Salvar como arquivo de texto simples (um valor por linha) para o código C ler
data_clean.to_csv('dados.csv', index=False, header=False)

print(f"Dados extraídos: {len(data_clean)} registros.")

In [ ]:
%%writefile centroides_iniciais.csv
50.500000
88.300000
41.500000
36.400000
15.800000
79.200000
86.100000
75.100000
35.400000
60.900000
78.000000
78.900000
66.300000
79.700000
73.900000
36.200000

# Versão Sequencial

/* kmeans_1d_naive.c
   K-means 1D (C99), implementação "naive":
   - Lê X (N linhas, 1 coluna) e C_iniciais (K linhas, 1 coluna).
   - Itera até MAX_ITER ou convergência (nenhuma mudança de cluster).
   - Salva C_finais e estatísticas (SSE, tempo).
*/

In [ ]:
%%writefile kmeans_1d_naive.c

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <time.h>

/* ---------- util CSV 1D: cada linha tem 1 número ---------- */
static int count_rows(const char *path){
    FILE *f = fopen(path, "r");
    if(!f){ fprintf(stderr,"Erro ao abrir %s\n", path); exit(1); }
    int rows=0; char line[8192];
    while(fgets(line,sizeof(line),f)){
        int only_ws=1;
        for(char *p=line; *p; p++){
            if(*p!=' ' && *p!='\t' && *p!='\n' && *p!='\r'){ only_ws=0; break; }
        }
        if(!only_ws) rows++;
    }
    fclose(f);
    return rows;
}

static double *read_csv_1col(const char *path, int *n_out){
    int R = count_rows(path);
    if(R<=0){ fprintf(stderr,"Arquivo vazio: %s\n", path); exit(1); }
    double *A = (double*)malloc((size_t)R * sizeof(double));
    if(!A){ fprintf(stderr,"Sem memoria para %d linhas\n", R); exit(1); }

    FILE *f = fopen(path, "r");
    if(!f){ fprintf(stderr,"Erro ao abrir %s\n", path); free(A); exit(1); }

    char line[8192];
    int r=0;
    while(fgets(line,sizeof(line),f)){
        int only_ws=1;
        for(char *p=line; *p; p++){
            if(*p!=' ' && *p!='\t' && *p!='\n' && *p!='\r'){ only_ws=0; break; }
        }
        if(only_ws) continue;

        /* aceita vírgula/ponto-e-vírgula/espaco/tab, pega o primeiro token numérico */
        const char *delim = ",; \t";
        char *tok = strtok(line, delim);
        if(!tok){ fprintf(stderr,"Linha %d sem valor em %s\n", r+1, path); free(A); fclose(f); exit(1); }
        A[r] = atof(tok);
        r++;
        if(r>R) break;
    }
    fclose(f);
    *n_out = R;
    return A;
}

static void write_assign_csv(const char *path, const int *assign, int N){
    if(!path) return;
    FILE *f = fopen(path, "w");
    if(!f){ fprintf(stderr,"Erro ao abrir %s para escrita\n", path); return; }
    for(int i=0;i<N;i++) fprintf(f, "%d\n", assign[i]);
    fclose(f);
}

static void write_centroids_csv(const char *path, const double *C, int K){
    if(!path) return;
    FILE *f = fopen(path, "w");
    if(!f){ fprintf(stderr,"Erro ao abrir %s para escrita\n", path); return; }
    for(int c=0;c<K;c++) fprintf(f, "%.6f\n", C[c]);
    fclose(f);
}

/* ---------- k-means 1D ---------- */
/* assignment: para cada X[i], encontra c com menor (X[i]-C[c])^2 */
static double assignment_step_1d(const double *X, const double *C, int *assign, int N, int K){
    double sse = 0.0;
    for(int i=0;i<N;i++){
        int best = -1;
        double bestd = 1e300;
        for(int c=0;c<K;c++){
            double diff = X[i] - C[c];
            double d = diff*diff;
            if(d < bestd){ bestd = d; best = c; }
        }
        assign[i] = best;
        sse += bestd;
    }
    return sse;
}

/* update: média dos pontos de cada cluster (1D)
   se cluster vazio, copia X[0] (estratégia naive) */
static void update_step_1d(const double *X, double *C, const int *assign, int N, int K){
    double *sum = (double*)calloc((size_t)K, sizeof(double));
    int *cnt = (int*)calloc((size_t)K, sizeof(int));
    if(!sum || !cnt){ fprintf(stderr,"Sem memoria no update\n"); exit(1); }

    for(int i=0;i<N;i++){
        int a = assign[i];
        cnt[a] += 1;
        sum[a] += X[i];
    }
    for(int c=0;c<K;c++){
        if(cnt[c] > 0) C[c] = sum[c] / (double)cnt[c];
        else           C[c] = X[0]; /* simples: cluster vazio recebe o primeiro ponto */
    }
    free(sum); free(cnt);
}

static void kmeans_1d(const double *X, double *C, int *assign,
                      int N, int K, int max_iter, double eps,
                      int *iters_out, double *sse_out)
{
    FILE *sse_file = fopen("sse_evolution_seq.csv", "w");
    double prev_sse = 1e300;
    double sse = 0.0;
    int it;
    for(it=0; it<max_iter; it++){
        sse = assignment_step_1d(X, C, assign, N, K);
        fprintf(sse_file, "%d,%.6f\n", it, sse);
        /* parada por variação relativa do SSE */
        double rel = fabs(sse - prev_sse) / (prev_sse > 0.0 ? prev_sse : 1.0);
        printf("Iteration %d, SSE: %f\n", it + 1, sse); // Added this line back
        if(rel < eps){ it++; break; }
        update_step_1d(X, C, assign, N, K);
        prev_sse = sse;

    }
    *iters_out = it;
    *sse_out = sse;
    fclose(sse_file);
}

/* ---------- main ---------- */
int main(int argc, char **argv){
    if(argc < 3){
        printf("Uso: %s dados.csv centroides_iniciais.csv [max_iter=50] [eps=1e-4] [assign.csv] [centroids.csv]\n", argv[0]);
        printf("Obs: arquivos CSV com 1 coluna (1 valor por linha), sem cabeçalho.\n");
        return 1;
    }
    const char *pathX = argv[1];
    const char *pathC = argv[2];
    int max_iter = (argc>3)? atoi(argv[3]) : 50;
    double eps   = (argc>4)? atof(argv[4]) : 1e-4;
    const char *outAssign   = (argc>5)? argv[5] : NULL;
    const char *outCentroid = (argc>6)? argv[6] : NULL;

    if(max_iter <= 0 || eps <= 0.0){
        fprintf(stderr,"Parâmetros inválidos: max_iter>0 e eps>0\n");
        return 1;
    }

    int N=0, K=0;
    double *X = read_csv_1col(pathX, &N);
    double *C = read_csv_1col(pathC, &K);
    int *assign = (int*)malloc((size_t)N * sizeof(int));
    if(!assign){ fprintf(stderr,"Sem memoria para assign\n"); free(X); free(C); return 1; }

    clock_t t0 = clock();
    int iters = 0; double sse = 0.0;
    kmeans_1d(X, C, assign, N, K, max_iter, eps, &iters, &sse);
    clock_t t1 = clock();
    double ms = 1000.0 * (double)(t1 - t0) / (double)CLOCKS_PER_SEC;

    printf("K-means 1D (naive)\n");
    printf("N=%d K=%d max_iter=%d eps=%g\n", N, K, max_iter, eps);
    printf("Iterações: %d | SSE final: %.6f | Tempo: %.1f ms\n", iters, sse, ms);

    write_assign_csv(outAssign, assign, N);
    write_centroids_csv(outCentroid, C, K);

    free(assign); free(X); free(C);
    return 0;
}


In [ ]:
%%shell

gcc -O2 -std=c99 kmeans_1d_naive.c -o kmeans_1d_naive -lm
./kmeans_1d_naive dados.csv centroides_iniciais.csv

In [ ]:
!apt-get update
!apt-get install -y openmpi-bin libopenmpi-dev

## Versão com MPI
kmeans_1d_naive.c
K-means 1D (C99), implementação "naive":
- Lê X (N linhas, 1 coluna) e C_iniciais (K linhas, 1 coluna).
- Itera até MAX_ITER ou convergência (nenhuma mudança de cluster).
- Salva C_finais e estatísticas (SSE, tempo).

In [ ]:
%%writefile kmeans_1d_mpi.c

#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <mpi.h>

static int count_rows(const char *path){
    FILE *f = fopen(path, "r");
    if(!f) return -1;
    int rows=0; char line[8192];
    while(fgets(line,sizeof(line),f)){
        int only_ws=1;
        for(char *p=line; *p; p++){
            if(*p!=' ' && *p!='\t' && *p!='\n' && *p!='\r'){ only_ws=0; break; }
        }
        if(!only_ws) rows++;
    }
    fclose(f);
    return rows;
}

static double *read_csv_1col(const char *path, int *n_out){
    int R = count_rows(path);
    if(R<=0) return NULL;
    double *A = (double*)malloc((size_t)R * sizeof(double));
    if(!A) return NULL;

    FILE *f = fopen(path, "r");
    if(!f){ free(A); return NULL; }

    char line[8192];
    int r=0;
    while(fgets(line,sizeof(line),f)){
        int only_ws=1;
        for(char *p=line; *p; p++){
            if(*p!=' ' && *p!='\t' && *p!='\n' && *p!='\r'){ only_ws=0; break; }
        }
        if(only_ws) continue;
        const char *delim = ",; \t";
        char *tok = strtok(line, delim);
        if(tok){
            A[r] = atof(tok);
            r++;
        }
        if(r>=R) break;
    }
    fclose(f);
    *n_out = R;
    return A;
}

static void write_assign_csv(const char *path, const int *assign, int N){
    if(!path) return;
    FILE *f = fopen(path, "w");
    if(!f) return;
    for(int i=0;i<N;i++) fprintf(f, "%d\n", assign[i]);
    fclose(f);
}

static void write_centroids_csv(const char *path, const double *C, int K){
    if(!path) return;
    FILE *f = fopen(path, "w");
    if(!f) return;
    for(int c=0;c<K;c++) fprintf(f, "%.6f\n", C[c]);
    fclose(f);
}

static double assignment_step_1d(const double *X, const double *C, int *assign, int N, int K){
    double sse = 0.0;
    for(int i=0;i<N;i++){
        int best = -1;
        double bestd = 1e300;
        for(int c=0;c<K;c++){
            double diff = X[i] - C[c];
            double d = diff*diff;
            if(d < bestd){ bestd = d; best = c; }
        }
        assign[i] = best;
        sse += bestd;
    }
    return sse;
}

int main(int argc, char **argv){
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);
    int N_global = 0, K = 0, max_iter = 50;
    double eps = 1e-4;
    double t_serial_ref = 0.0; // Para calculo de metricas
    double *X_full = NULL;
    double *C = NULL;

    if(rank == 0) {
        if(argc > 3) max_iter = atoi(argv[3]);
        if(argc > 4) eps = atof(argv[4]);
        if(argc > 5) t_serial_ref = atof(argv[5]); // Leitura do tempo serial passado por argumento

        X_full = read_csv_1col(argv[1], &N_global);
        C = read_csv_1col(argv[2], &K);
        if(!X_full || !C) MPI_Abort(MPI_COMM_WORLD, 1);
    }

    MPI_Bcast(&N_global, 1, MPI_INT, 0, MPI_COMM_WORLD);
    MPI_Bcast(&K, 1, MPI_INT, 0, MPI_COMM_WORLD);
    MPI_Bcast(&max_iter, 1, MPI_INT, 0, MPI_COMM_WORLD);
    MPI_Bcast(&eps, 1, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    if(rank != 0) C = (double*)malloc((size_t)K * sizeof(double));
    MPI_Bcast(C, K, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    int *sendCounts = (int*)malloc((size_t)size * sizeof(int));
    int *displs = (int*)malloc((size_t)size * sizeof(int));
    int rem = N_global % size;
    int sum = 0;
    for(int i = 0; i < size; i++) {
        sendCounts[i] = N_global / size + (i < rem ? 1 : 0);
        displs[i] = sum;
        sum += sendCounts[i];
    }

    int N_local = sendCounts[rank];
    double *X_local = (double*)malloc((size_t)N_local * sizeof(double));
    int *assign_local = (int*)malloc((size_t)N_local * sizeof(int));

    MPI_Scatterv(X_full, sendCounts, displs, MPI_DOUBLE, X_local, N_local, MPI_DOUBLE, 0, MPI_COMM_WORLD);

    double *sum_local = (double*)calloc((size_t)K, sizeof(double));
    int *count_local = (int*)calloc((size_t)K, sizeof(int));
    double *sum_global = (double*)calloc((size_t)K, sizeof(double));
    int *count_global = (int*)calloc((size_t)K, sizeof(int));

    double prev_sse = 1e300, sse = 0.0;
    int it;
    double t_comm_accum = 0.0;
    double comm_start, comm_end;

    MPI_Barrier(MPI_COMM_WORLD);
    double start_time = MPI_Wtime();

    for (it=0; it<max_iter; it++){
        sse = assignment_step_1d(X_local, C, assign_local, N_local, K);
        double global_sse;

    comm_start = MPI_Wtime();
    MPI_Allreduce(&sse, &global_sse, 1, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);
    comm_end = MPI_Wtime();
    t_comm_accum += (comm_end - comm_start);

    sse = global_sse;
    double rel = fabs(sse - prev_sse) / (prev_sse > 0.0 ? prev_sse : 1.0);
    if(rel < eps){ it++; break; }

    memset(sum_local, 0, (size_t)K * sizeof(double));
    memset(count_local, 0, (size_t)K * sizeof(int));

    for(int i=0; i<N_local; i++){
        int a = assign_local[i];
        count_local[a] += 1;
        sum_local[a] += X_local[i];
    }

    comm_start = MPI_Wtime();
    MPI_Allreduce(sum_local, sum_global, K, MPI_DOUBLE, MPI_SUM, MPI_COMM_WORLD);
    MPI_Allreduce(count_local, count_global, K, MPI_INT, MPI_SUM, MPI_COMM_WORLD);
    comm_end = MPI_Wtime();
    t_comm_accum += (comm_end - comm_start);

    for(int c=0; c<K; c++){
        if(count_global[c] > 0) C[c] = sum_global[c] / (double)count_global[c];
        else                   C[c] = X_local[0];
    }
    prev_sse = sse;
}

    double total_time = MPI_Wtime() - start_time;
    double calc_time = total_time - t_comm_accum;

    if(rank == 0) {
        // Calculo das metricas em C
        double speedup = 0.0;
        double efficiency = 0.0;

    if (size == 1) {
        speedup = 1.0;
        efficiency = 1.0;
    } else if (t_serial_ref > 0.0) {
        speedup = t_serial_ref / total_time;
        efficiency = speedup / (double)size;
    }

    // Output com metricas calculadas
    printf("RESULT, %d, %d, %d, %.6f, %.6f, %.6f, %.2f, %.2f\n",
           size, N_global, it, total_time, t_comm_accum, calc_time, speedup, efficiency);

    free(X_full);
    }
    free(C); free(X_local); free(assign_local);
    free(sum_local); free(count_local); free(sum_global); free(count_global);
    free(sendCounts); free(displs);
    MPI_Finalize();
    return 0;
}

In [ ]:
%%bash
echo "Compilando..."
mpicc -O2 -std=c99 kmeans_1d_mpi.c -o kmeans_1d_mpi -lm
echo "--------------------------------------------------------------------------------"
echo "P     Iter   Total(s)    Comm(s)     Calc(s)     Speedup     Eficiência"
echo "--------------------------------------------------------------------------------"

T_SERIAL=0.0

for P in 1 2 4; do
OUTPUT=$(mpirun --allow-run-as-root --oversubscribe -np $P ./kmeans_1d_mpi dados.csv centroides_iniciais.csv 50 1e-4 $T_SERIAL)

RES_LINE=$(echo "$OUTPUT" | grep "RESULT")

ITER=$(echo $RES_LINE | awk -F, '{print $4}')
TOTAL=$(echo $RES_LINE | awk -F, '{print $5}')
COMM=$(echo $RES_LINE | awk -F, '{print $6}')
CALC=$(echo $RES_LINE | awk -F, '{print $7}')
SPEEDUP=$(echo $RES_LINE | awk -F, '{print $8}')
EFF=$(echo $RES_LINE | awk -F, '{print $9}')

if [ "$P" -eq "1" ]; then
    T_SERIAL=$TOTAL
fi
    printf "%-5d %-6d %-11.4f %-11.4f %-11.4f %-11.2f %-11.2f\n" $P $ITER $TOTAL $COMM $CALC $SPEEDUP $EFF
done

